# Testes extras de robustez — mesmo desenho do Experimento A (BFI-44 + IPA, N=300)

Três testes, cada um numa célula independente (rode na ordem que quiser):

| Teste | Modelo | Temperatura | Seed | Saída |
|---|---|---|---|---|
| **A. Outra semente** | llama3.1:8b | 0,7 | **456** | `resultados/robustez/exp_a_seed456_bfi44_8b.csv` |
| **B. Temperatura 1,0** | llama3.1:8b | **1,0** | 123 | `resultados/temperatura/exp_a_t10_bfi44_8b.csv` |
| **C. Modelos abertos** | **qwen2.5:7b, mistral:7b, gemma3:12b** | 0,7 | 123 | `resultados/modelos_locais/exp_a_<modelo>.csv` |

- **A** sorteia perfis **diferentes** dos do Experimento A: testa se os resultados dependem do sorteio específico.
- **B** completa a curva de temperatura (0,0 / 0,3 / 0,7 / 1,0) com os **mesmos perfis** do Experimento A.
- **C** usa os **mesmos perfis** do Experimento A em outros modelos abertos que cabem na GPU T4. Para esses modelos o teto de resposta é 1.536 tokens (o llama usou 512), porque alguns modelos repetem o texto dos itens do BFI-44; o log avisa se alguma resposta for cortada.

Tempo aproximado: ~1h30 por rodada com o llama3.1:8b; os outros modelos podem levar mais. **Se a sessão cair, rode de novo a mesma célula: ela continua de onde parou, sem duplicar perfis.** Cada rodada é copiada para o Drive a cada 25 agentes e ao final.

## Antes de rodar

1. `Ambiente de execução → Alterar tipo → GPU T4`.
2. Substitua no Drive (`Meu Drive/tcc/projeto-tcc`) o **`src/geracao/gerar_agentes.py`** pela versão nova (que aceita `--max-tokens` e registra no log erros de conexão com o Ollama).

---
## Passo 1 — GPU e Ollama

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU disponível:', r.stdout.strip()) if r.returncode == 0 else print('ATENÇÃO: GPU não detectada (vai rodar em CPU, muito mais lento).')

In [ ]:
!apt-get install -y -q zstd
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
import subprocess, time, os, requests
OLLAMA_BIN = '/usr/local/bin/ollama'

def garantir_ollama_rodando():
    try:
        if requests.get('http://localhost:11434', timeout=5).ok:
            return
    except Exception:
        pass
    print('Iniciando o servidor do Ollama...')
    subprocess.Popen([OLLAMA_BIN, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    time.sleep(5)

def garantir_modelo(modelo):
    r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
    if modelo not in r.stdout:
        print(f'Baixando {modelo} (alguns minutos)...')
        subprocess.run([OLLAMA_BIN, 'pull', modelo], check=True)

garantir_ollama_rodando()

---
## Passo 2 — Copiar o projeto do Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_PROJETO = '/content/drive/MyDrive/tcc/projeto-tcc'   # ajuste se a pasta estiver em outro lugar
LOCAL_PROJETO = '/content/projeto-tcc'
PASTAS = ['robustez', 'temperatura', 'modelos_locais']

if not os.path.exists(DRIVE_PROJETO):
    raise FileNotFoundError(f'Pasta não encontrada: {DRIVE_PROJETO}')
if os.path.exists(LOCAL_PROJETO):
    shutil.rmtree(LOCAL_PROJETO)
shutil.copytree(DRIVE_PROJETO, LOCAL_PROJETO)
for p in PASTAS:
    os.makedirs(f'{LOCAL_PROJETO}/resultados/{p}', exist_ok=True)

if 'ERRO INESPERADO' not in open(f'{LOCAL_PROJETO}/src/geracao/gerar_agentes.py', encoding='utf-8').read():
    raise RuntimeError('O gerar_agentes.py do Drive é uma versão antiga (sem o registro de erros do Ollama). Substitua-o no Drive e rode esta célula de novo.')
print('Projeto pronto em', LOCAL_PROJETO)

---
## Passo 3 — Função que roda (ou retoma) uma rodada

Calcula quantos agentes faltam para chegar a 300, roda só esses e copia o CSV e o log para o Drive (a cada 25 agentes e ao final).

In [ ]:
import shutil

N_TOTAL = 300

def salvar_no_drive(caminho):
    pasta = os.path.dirname(caminho).replace(LOCAL_PROJETO, DRIVE_PROJETO)
    os.makedirs(pasta, exist_ok=True)
    for ext in ('.csv', '.log'):
        arq = caminho[:-4] + ext
        if os.path.exists(arq):
            shutil.copy2(arq, pasta)

def contar(caminho):
    if not os.path.exists(caminho):
        return 0
    with open(caminho, encoding='utf-8') as f:
        return max(0, sum(1 for _ in f) - 1)

def rodar(saida, modelo, temperatura=0.7, seed=123, max_tokens=None, max_reinicios=3):
    caminho = f'{LOCAL_PROJETO}/{saida}'
    # Traz do Drive uma rodada já começada (retomada depois de uma queda da sessão)
    no_drive = f'{DRIVE_PROJETO}/{saida}'
    for ext in ('.csv', '.log'):
        if os.path.exists(no_drive[:-4] + ext) and contar(no_drive[:-4] + '.csv') > contar(caminho):
            shutil.copy2(no_drive[:-4] + ext, caminho[:-4] + ext)
    # Se o Ollama cair no meio, o gerar_agentes.py para e anota o erro no log;
    # aqui o servidor é reiniciado e a rodada continua de onde parou.
    for reinicio in range(max_reinicios + 1):
        garantir_ollama_rodando()
        garantir_modelo(modelo)
        faltam = N_TOTAL - contar(caminho)
        print(f'{saida}: {contar(caminho)} já gerados, faltam {faltam}.')
        if faltam <= 0:
            break
        env = {**os.environ, 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}
        cmd = ['python', 'src/geracao/gerar_agentes.py', '--n', str(faltam), '--output', saida, '--modo', 'multi',
               '--seed', str(seed), '--modelo', modelo, '--temperatura', str(temperatura)]
        if max_tokens:
            cmd += ['--max-tokens', str(max_tokens)]
        proc = subprocess.Popen(cmd, cwd=LOCAL_PROJETO, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        gerados = 0
        for linha in proc.stdout:
            print(linha, end='')
            if 'agente_id=' in linha:
                gerados += 1
                if gerados % 25 == 0:
                    salvar_no_drive(caminho)
        proc.wait()
        salvar_no_drive(caminho)
        if contar(caminho) >= N_TOTAL:
            break
        print(f'\n>>> Rodada parou com {contar(caminho)}/{N_TOTAL}. Reiniciando o Ollama e retomando ({reinicio + 1}/{max_reinicios})...')
        subprocess.run(['pkill', '-f', 'ollama serve'])
        time.sleep(5)
    print(f'Salvo no Drive: {no_drive} ({contar(caminho)}/{N_TOTAL} agentes)')
    if contar(caminho) < N_TOTAL:
        print('ATENÇÃO: não completou. Veja o fim do log acima (linhas "erro na chamada" / "PARADA" / "ERRO INESPERADO").')

---
## Teste A — Outra semente (seed 456, temperatura 0,7)

In [ ]:
rodar('resultados/robustez/exp_a_seed456_bfi44_8b.csv', 'llama3.1:8b', temperatura=0.7, seed=456)

---
## Teste B — Temperatura 1,0 (seed 123, mesmos perfis do Experimento A)

In [ ]:
rodar('resultados/temperatura/exp_a_t10_bfi44_8b.csv', 'llama3.1:8b', temperatura=1.0, seed=123)

---
## Teste C — Modelos abertos (temperatura 0,7, seed 123, mesmos perfis do Experimento A)

Um modelo por célula. Antes de cada rodada completa, a célula de teste rápido mostra se o modelo responde no formato certo (vale olhar antes de gastar 1h30+).

In [ ]:
# Teste rápido: 2 agentes de cada modelo numa pasta à parte (não entra na análise)
for modelo, apelido in [('qwen2.5:7b', 'qwen25_7b'), ('mistral:7b', 'mistral_7b'), ('gemma3:12b', 'gemma3_12b')]:
    garantir_ollama_rodando(); garantir_modelo(modelo)
    saida = f'resultados/modelos_locais/teste_{apelido}.csv'
    if os.path.exists(f'{LOCAL_PROJETO}/{saida}'):
        os.remove(f'{LOCAL_PROJETO}/{saida}')
    env = {**os.environ, 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}
    r = subprocess.run(['python', 'src/geracao/gerar_agentes.py', '--n', '2', '--output', saida, '--modo', 'multi', '--seed', '123',
                        '--modelo', modelo, '--max-tokens', '1536', '--resumo-cada', '2'],
                       cwd=LOCAL_PROJETO, env=env, capture_output=True, text=True)
    print(f'===== {modelo}'); print(r.stdout[-1500:])

In [ ]:
rodar('resultados/modelos_locais/exp_a_qwen25_7b.csv', 'qwen2.5:7b', max_tokens=1536)

In [ ]:
rodar('resultados/modelos_locais/exp_a_mistral_7b.csv', 'mistral:7b', max_tokens=1536)

In [ ]:
rodar('resultados/modelos_locais/exp_a_gemma3_12b.csv', 'gemma3:12b', max_tokens=1536)

---
## Depois

Baixe do Drive as pastas `resultados/robustez`, `resultados/temperatura` (só o `exp_a_t10_*`) e `resultados/modelos_locais` para o projeto no computador. A análise completa roda localmente com `src/analise/analise_temperatura_modelos.py`.